# Stateful operators

Stateful operators bring the real fun to Kafi Streams.

The most interesting operators are of course `join()` and `join_pred()` (equi join and general/non-equi join on a predicate) and `group_by_agg()` (group by + aggregate).

`agg()` is just a special case of `group_by_agg()` (obviously, without grouping).

Kafi Streams also supports the stateful set operators `distinct()`, `union()`, `intersect()` and `minus()`.

The `collapse()` operator is key for handling the [stream/table duality](../duality.ipynb).

## Overview

[Preparation](#prep)

* [join()](#join-operator)
* [join_pred()](#join_pred-operator)
* [group_by_agg()](#group_by_agg-operator)
  * [group_by_sum()](#group_by_sum-operator)
  * [group_by_max()](#group_by_max-operator)
  * [group_by_min()](#group_by_min-operator)
  * [group_by_avg()](#group_by_avg-operator)
  * [group_by_count()](#group_by_count-operator)
  * [agg()](#agg-operator)
    * [sum()](#sum-operator)
    * [max()](#max-operator)
    * [min()](#min-operator)
    * [avg()](#avg-operator)
    * [count()](#count-operator)
* [distinct()](#distinct-operator)
* [union()](#union-operator)
* [intersect()](#intersect-operator)
* [minus()](#minus-operator)
* [collapse()](#collapse-operator)


---
<a id="prep"></a>
## Preparation

Before we start off, we first prepare for the examples to follow:

In [1]:
!pip install -r ../requirements.txt

import sys
sys.path.insert(1, "../")
sys.path.insert(1, "../../..")

from kafi.streams.topologynode import TopologyNode as Tn

from generators import ClickGenerator, CustomerGenerator
click_generator = ClickGenerator()
customer_generator = CustomerGenerator()

click_source_str = "clicks"
customer_source_str = "customers"


---
<a id="join-operator"></a>
## join()

Probably the most popular stateful operator - `join()` is the equi join operator of Kafi Streams:
```python
def join(self, right_tn, left_key_fun, right_key_fun, project_fun, **kwargs):
"""Equi-join two topology nodes based on keys from both sides.

Args:
    right_tn: the other topology node to join with
    left_key_fun: l_r -> key - get the key of the left record l_r
    right_key_fun: r_r -> key - get the key of the right record r_r
    project_fun: (l_r, r_r) -> r - projection function for left record l_r and right record r_r
    **kwargs: passed through to the underlying node(s)
Returns:
    tn: the newly created topology node of the operator"""
```

Note that the `left_key_fun` and `right_key_fun` can in principle select *anything* from the incoming records. As Kafi Streams is not tied to Kafka but just receives a Python dictionary, there are no restrictions to e.g. Kafka keys as in Kafka Streams.

Here is an example:

In [ ]:
click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"], "ts": r["value"]["ts"]})
)

customer_tn = (
    Tn.source(customer_source_str)
    #
    .map(lambda r: {"id": r["value"]["id"], "name": r["value"]["name"]})
)

tn = Tn.build(
    click_tn
    ###
    # join() operator - equi join based on customer_id (clicks) and id (customers) and project customer_id, view_time and name 
    ###
    .join(customer_tn,
          left_key_fun=lambda l_r: l_r["customer_id"],
          right_key_fun=lambda r_r: r_r["id"],
          project_fun=lambda l_r, r_r: {"customer_id": l_r["customer_id"],
                                        "view_time": l_r["view_time"],
                                        "ts": l_r["ts"],
                                        "name": r_r["name"]})
)

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618958910}},
                      {'key': None, 'value': {'customer_id': 24, 'view_time': 76, 'ts': 1786618968910}},
                      {'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

customer_input_m_list = [{'key': '42', 'value': {'id': 42, 'name': 'Betty Graham MD'}},
                         {'key': '4711', 'value': {'id': 4711, 'name': 'Frank Frank'}}]
print("\nInput (customers):")
for m in customer_input_m_list:
    print(m)

output_m_list = tn.process({click_source_str: click_input_m_list, customer_source_str: customer_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


In this example topology, we join the clicks and customers as before in the [Quickstart](../quickstart.ipynb).

Here is the Mermaid version of the topology:
```mermaid
graph TD
a22fd660-1947-429b-a06e-db734ba723ca[source_clicks] --> bb9deb47-1296-460e-8346-2ab45e62d237[map_op]
0a9a77fb-bf6f-4518-96a0-31f76dd32ff9[source_customers] --> bb101b8f-cd20-4ea6-ae74-cf13e102bbce[map_op]
bb9deb47-1296-460e-8346-2ab45e62d237[map_op] --> 2bd2f663-8a95-48fb-a462-86fa43bef0d1[join_op]
bb101b8f-cd20-4ea6-ae74-cf13e102bbce[map_op] --> 2bd2f663-8a95-48fb-a462-86fa43bef0d1[join_op]
```

The example data consists of three clicks and two customers. Two of the clicks match a customer from the right side (`customer_id` = `42`) and thus we receive both clicks (joined with the `name`) in the output.

---
<a id="join_pred-operator"></a>
## join_pred()

While `join()` might be the most popular stateful operator, Kafi Streams also offers a general/non-equi join with the `join_pred()` operator.

Whereas with `join()`, you can only select a key from each side of the join, `join_pred()` allows you to specify any predicate:
```python
def join_pred(self, right_tn, predicate_fun, project_fun, **kwargs):
    """Join two topology nodes based on an arbitrary predicate.
    
    Args:
        right_tn: the other topology node to join with
        predicate_fun: (l_r, r_r) -> bool - the join predicate for left record l_r and right record r_r
        project_fun: (l_r, r_r) -> r - projection function for left record l_r and right record r_r
        **kwargs: passed through to the underlying node(s)
    Returns:
        tn: the newly created topology node of the operator"""
```

As in the database world as well, the non-equi join operator `join_pred()` is, on the one hand, more flexible than the equi join `join()`, but on the other hand much less performant:
* `join()` is based on hash maps (time complexity: `O(1)` per row)
* `join_pred()` cannot use hash maps (time complexity:`O(N)` per row)

So whenever you can, use `join()`. Use `join_pred()` only if you need the extra flexibility or the predicate can be calculated quickly enough (e.g. in case one of the two sides of the join always has only few elements).

An example is in order for the `join_pred()` operator as well:

In [ ]:
click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"], "ts": r["value"]["ts"]})
)

customer_tn = (
    Tn.source(customer_source_str)
    #
    .map(lambda r: {"id": r["value"]["id"], "name": r["value"]["name"]})
)

tn = Tn.build(
    click_tn
    ###
    # join_pred() operator - join by customer ID but also only if view_time > 60
    ###
    .join_pred(customer_tn,
               predicate_fun=lambda l_r, r_r: l_r["customer_id"] == r_r["id"] and l_r["view_time"] > 60,
               project_fun=lambda l_r, r_r: {"customer_id": l_r["customer_id"],
                                             "view_time": l_r["view_time"],
                                             "ts": l_r["ts"],
                                             "name": r_r["name"]})
)

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618958910}},
                      {'key': None, 'value': {'customer_id': 24, 'view_time': 76, 'ts': 1786618968910}},
                      {'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

customer_input_m_list = [{'key': '42', 'value': {'id': 42, 'name': 'Betty Graham MD'}},
                         {'key': '4711', 'value': {'id': 4711, 'name': 'Frank Frank'}}]
print("\nInput (customers):")
for m in customer_input_m_list:
    print(m)

output_m_list = tn.process({click_source_str: click_input_m_list, customer_source_str: customer_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


Except for the change from `join()` to `join_pred()`, the topology is the same as in the previous example for `join()`.

However, this time, we do not only match the customer IDs but also add the condition that the `view_time` of the click must be greater than `60`.

Here is the Mermaid version of the topology:
```mermaid
graph TD
a22fd660-1947-429b-a06e-db734ba723ca[source_clicks] --> bb9deb47-1296-460e-8346-2ab45e62d237[map_op]
0a9a77fb-bf6f-4518-96a0-31f76dd32ff9[source_customers] --> bb101b8f-cd20-4ea6-ae74-cf13e102bbce[map_op]
bb9deb47-1296-460e-8346-2ab45e62d237[map_op] --> 2bd2f663-8a95-48fb-a462-86fa43bef0d1[join_pred_op]
bb101b8f-cd20-4ea6-ae74-cf13e102bbce[map_op] --> 2bd2f663-8a95-48fb-a462-86fa43bef0d1[join_pred_op]
```

The example data is the same as in the previous example for `join()` - but the output is different, because even though the second click of customer `42` does match the customer ID of one of the customers, its `view_time` is not greater than `60`. Thus we only receive one output record.

---
<a id="group_by_agg-operator"></a>
## group_by_agg()

The `group_by_agg()` operator is, as its name implies, a combination of grouping by some key and an arbitrary aggregation:
```python
def group_by_agg(self, key_fun, value_fun, agg_fun, agg_initial_any, project_fun, **kwargs):
    """Group values by key and aggregate the values.
    
    Args:
        key_fun: r -> key_any - function to get the key
        value_fun: r -> value_any - function to get the value to aggregate
        agg_fun: (agg any, value_any) -> aggregate function; agg_any: running aggregate, value_any: current value
        agg_initial_any: initial aggregate
        project_fun: (key_any, agg_any) -> r - projection function
        **kwargs: passed through to the underlying node(s)
    Returns:
        tn: the newly created topology node of the operator"""

```

Here is an example:


In [ ]:
click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"]})
)

tn = Tn.build(
    click_tn
    ###
    # group_by_agg() operator: 
    #   * group by customer_id,
    #   * select the value view_time
    #   * aggregation:
    #     * view_times: the collected view times
    #     * sum_view_times: the sum of the view times
    #   * projection: customer_id (key), view_times and sum_view_times (aggregation)
    ###
    .group_by_agg(key_fun=lambda r: r["customer_id"],
                  value_fun=lambda r: r["view_time"],
                  agg_fun=lambda agg_any, value_any: {"view_times": agg_any["view_times"] + [value_any],
                                                      "sum_view_times": agg_any["sum_view_times"] + value_any},
                  agg_initial_any={"view_times": [],
                                   "sum_view_times": 0},
                  project_fun=lambda key_any, agg_any: {"customer_id": key_any,
                                                        "view_times": agg_any["view_times"],
                                                        "sum_view_times": agg_any["sum_view_times"]})
)

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618958910}},
                      {'key': None, 'value': {'customer_id': 24, 'view_time': 76, 'ts': 1786618968910}},
                      {'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

output_m_list = tn.process({click_source_str: click_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


In the example topology, we group by `customer_id` and aggregate the `view_time` of our clickstream input.

The actual aggregation function does two things:
1. It collects each `view_time` per customers in the field `view_times`.
2. It sums up each `view_time` per customer in the field `sum_view_times`.

The projection function returns records with three fields:
1. `customer_id`: the group/key
2. `view_times`: the collected view times
3. `sum_view_times`: the sum of the view times

Here is the graphical representation of the example topology:
```mermaid
graph TD
0d1c9c98-1359-48fc-9a53-678c46c02a6d[map_op] --> 4fb0a263-f417-4060-a85d-2fdba48ee8b4[group_by_agg_op]
6b7e4b4f-b23c-4e3a-9bd4-102c88373ebf[source_clicks] --> 0d1c9c98-1359-48fc-9a53-678c46c02a6d[map_op]
```

In the example data, customer `42` has two clicks: one with a `view_time` of `67` and one with `23`. Hence, after the aggregation, the `view_times` are `[67, 23]` and the sum is `90`.


<a id="group_by_sum-operator"></a>
### group_by_sum()

`group_by_sum()` is syntactic sugar for `group_by_agg()` where the aggregation function sums up the selected values:
```python
def group_by_sum(self, key_fun, value_fun, project_fun, sum_initial_any=0, **kwargs):
    """Sum values per key.
    
    Args:
        key_fun: r -> key_any - function to get the key
        value_fun: r -> value_any - function to get the value to aggregate
        project_fun: (key_any, agg_any) -> r - projection function
        sum_initial_any: initial sum (default: 0)
        **kwargs: passed through to the underlying node(s)
    Returns:
        tn: the newly created topology node of the operator"""
```

Here is an example:


In [ ]:
click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"]})
)

tn = Tn.build(
    click_tn
    ###
    # group_by_sum() operator: 
    #   * group by customer_id,
    #   * select the value view_time
    #   * aggregation:
    #     * sum_view_times: the sum of the view times
    #   * projection: customer_id (key), sum_view_times (aggregation)
    ###
    .group_by_sum(key_fun=lambda r: r["customer_id"],
                  value_fun=lambda r: r["view_time"],
                  project_fun=lambda key_any, agg_any: {"customer_id": key_any,
                                                        "sum_view_times": agg_any})
)

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618958910}},
                      {'key': None, 'value': {'customer_id': 24, 'view_time': 76, 'ts': 1786618968910}},
                      {'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

output_m_list = tn.process({click_source_str: click_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


<a id="group_by_max-operator"></a>
### group_by_max()

`group_by_max()` is syntactic sugar for `group_by_agg()` where the aggregation function calculates the maximum of the selected values:
```python
def group_by_max(self, key_fun, value_fun, project_fun, max_initial_any=0, **kwargs):
    """Maximum of values per key.
    
    Args:
        key_fun: r -> key_any - function to get the key
        value_fun: r -> value_any - function to get the value to aggregate
        project_fun: (key_any, agg_any) -> r - projection function
        max_initial_any: initial maximum (default: 0)
        **kwargs: passed through to the underlying node(s)
    Returns:
        tn: the newly created topology node of the operator"""

```

Here is an example:


In [ ]:
click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"]})
)

tn = Tn.build(
    click_tn
    ###
    # group_by_max() operator: 
    #   * group by customer_id,
    #   * select the value view_time
    #   * aggregation:
    #     * max_view_time: the maximum of the view times
    #   * projection: customer_id (key), max_view_time (aggregation)
    ###
    .group_by_max(key_fun=lambda r: r["customer_id"],
                  value_fun=lambda r: r["view_time"],
                  project_fun=lambda key_any, agg_any: {"customer_id": key_any,
                                                        "max_view_time": agg_any})
)

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618958910}},
                      {'key': None, 'value': {'customer_id': 24, 'view_time': 76, 'ts': 1786618968910}},
                      {'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

output_m_list = tn.process({click_source_str: click_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


<a id="group_by_min-operator"></a>
### group_by_min()

`group_by_min()` is syntactic sugar for `group_by_agg()` where the aggregation function calculates the minimum of the selected values:
```python
    def group_by_min(self, key_fun, value_fun, project_fun, min_initial_any=sys.maxsize, **kwargs):
    """Minimum of values per key.

    Args:
        key_fun: r -> key_any - function to get the key
        value_fun: r -> value_any - function to get the value to aggregate
        project_fun: (key_any, agg_any) -> r - projection function
        min_initial_any: initial minimum (default: sys.maxsize)
        **kwargs: passed through to the underlying node(s)
    Returns:
        tn: the newly created topology node of the operator"""
```

Here is an example:

In [ ]:
click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"]})
)

tn = Tn.build(
    click_tn
    ###
    # group_by_min() operator: 
    #   * group by customer_id,
    #   * select the value view_time
    #   * aggregation:
    #     * min_view_time: the minimum of the view times
    #   * projection: customer_id (key), min_view_time (aggregation)
    ###
    .group_by_min(key_fun=lambda r: r["customer_id"],
                  value_fun=lambda r: r["view_time"],
                  project_fun=lambda key_any, agg_any: {"customer_id": key_any,
                                                        "min_view_time": agg_any})
)

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618958910}},
                      {'key': None, 'value': {'customer_id': 24, 'view_time': 76, 'ts': 1786618968910}},
                      {'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

output_m_list = tn.process({click_source_str: click_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


<a id="group_by_avg-operator"></a>
### group_by_avg()

`group_by_avg()` is syntactic sugar for `group_by_agg()` where the aggregation function calculates the average of the selected values:
```python
def group_by_avg(self, key_fun, value_fun, project_fun, **kwargs):
    """Average of values per key.
    
    Args:
        key_fun: r -> key_any - function to get the key
        value_fun: r -> value_any - function to get the value to aggregate
        project_fun: (key_any, agg_any) -> r - projection function
        **kwargs: passed through to the underlying node(s)
    Returns:
        tn: the newly created topology node of the operator"""

```

Here is an example:


In [ ]:
click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"]})
)

tn = Tn.build(
    click_tn
    ###
    # group_by_avg() operator: 
    #   * group by customer_id,
    #   * select the value view_time
    #   * aggregation:
    #     * avg_view_time: the average of the view times
    #   * projection: customer_id (key), avg_view_time (aggregation)
    ###
    .group_by_avg(key_fun=lambda r: r["customer_id"],
                  value_fun=lambda r: r["view_time"],
                  project_fun=lambda key_any, agg_any: {"customer_id": key_any,
                                                        "avg_view_time": agg_any})
)

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618958910}},
                      {'key': None, 'value': {'customer_id': 24, 'view_time': 76, 'ts': 1786618968910}},
                      {'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

output_m_list = tn.process({click_source_str: click_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


<a id="group_by_count-operator"></a>
### group_by_count()

`group_by_count()` is syntactic sugar for `group_by_agg()` where the aggregation function counts the grouped values.
```python
def group_by_count(self, key_fun, project_fun, **kwargs):
    """Count records per key.
    
    Args:
        key_fun: r -> key_any - function to get the key
        project_fun: (key_any, agg_any) -> r - projection function
        **kwargs: passed through to the underlying node(s)
    Returns:
        tn: the newly created topology node of the operator"""
```

Here is an example:


In [ ]:
click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"]})
)

tn = Tn.build(
    click_tn
    ###
    # group_by_count() operator: 
    #   * group by customer_id,
    #   * aggregation:
    #     * count_clicks: count of the click records
    #   * projection: customer_id (key), count_view_times (aggregation)
    ###
    .group_by_count(key_fun=lambda r: r["customer_id"],
                    project_fun=lambda key_any, agg_any: {"customer_id": key_any,
                                                          "count_clicks": agg_any})
)

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618958910}},
                      {'key': None, 'value': {'customer_id': 24, 'view_time': 76, 'ts': 1786618968910}},
                      {'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

output_m_list = tn.process({click_source_str: click_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


<a id="agg-operator"></a>
### agg()

`agg()` is syntactic sugar for `group_by_agg()` which just aggregates (there is only one group/key),
```python
def agg(self, value_fun, agg_fun, agg_initial_any, project_fun, **kwargs):
    """Aggregate values.
    
    Args:
        value_fun: r -> value_any - function to get the value to aggregate
        agg_fun: (agg any, value_any) -> aggregate function; agg_any: running aggregate, value_any: current value
        agg_initial_any: initial aggregate
        project_fun: (key_any, agg_any) -> r - projection function
        **kwargs: passed through to the underlying node(s)
    Returns:
        tn: the newly created topology node of the operator"""
```

Here is an example:


In [ ]:
click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"]})
)

tn = Tn.build(
    click_tn
    ###
    # agg() operator: 
    #   * select the value view_time
    #   * aggregation:
    #     * view_times: the collected view times
    #     * sum_view_times: the sum of the view times
    #   * projection: customer_id (key), view_times and sum_view_times (aggregation)
    ###
    .agg(value_fun=lambda r: r["view_time"],
         agg_fun=lambda agg_any, value_any: {"view_times": agg_any["view_times"] + [value_any],
                                             "sum_view_times": agg_any["sum_view_times"] + value_any},
         agg_initial_any={"view_times": [],
                          "sum_view_times": 0},
         project_fun=lambda agg_any: {"view_times": agg_any["view_times"],
                                      "sum_view_times": agg_any["sum_view_times"]})
)

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618958910}},
                      {'key': None, 'value': {'customer_id': 24, 'view_time': 76, 'ts': 1786618968910}},
                      {'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

output_m_list = tn.process({click_source_str: click_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


<a id="sum-operator"></a>
### sum()

`sum()` is syntactic sugar for `agg()` where the aggregation function sums up the selected values:
```python
def sum(self, value_fun, project_fun=lambda agg_any: agg_any, sum_initial_any=0, **kwargs):
    """Sum values.
    
    Args:
        value_fun: r -> value_any - function to get the value to aggregate
        project_fun: (key_any, agg_any) -> r - projection function
        sum_initial_any: initial sum (default: 0)
        **kwargs: passed through to the underlying node(s)
    Returns:
        tn: the newly created topology node of the operator"""
```

Here is an example:


In [ ]:
click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"]})
)

tn = Tn.build(
    click_tn
    ###
    # sum() operator: 
    #   * select the value view_time
    #   * aggregation:
    #     * the sum of the view times
    #   * projection: default = identity function of the aggregation
    ###
    .sum(value_fun=lambda r: r["view_time"])
)

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618958910}},
                      {'key': None, 'value': {'customer_id': 24, 'view_time': 76, 'ts': 1786618968910}},
                      {'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

output_m_list = tn.process({click_source_str: click_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


<a id="max-operator"></a>
### max()

`max()` is syntactic sugar for `agg()` where the aggregation function calculates the maximum of the selected values:
```python
def max(self, value_fun, project_fun=lambda agg_any: agg_any, max_initial_any=0, **kwargs):
    """Maximum of values.
    
    Args:
        value_fun: r -> value_any - function to get the value to aggregate
        project_fun: (key_any, agg_any) -> r - projection function
        max_initial_any: initial maximum (default: 0)
        **kwargs: passed through to the underlying node(s)
    Returns:
        tn: the newly created topology node of the operator"""
```

Here is an example:


In [ ]:
click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"]})
)

tn = Tn.build(
    click_tn
    ###
    # max() operator: 
    #   * select the value view_time
    #   * aggregation:
    #     * the maximum of the view times
    #   * projection: default = identity function of the aggregation
    ###
    .max(value_fun=lambda r: r["view_time"])
)

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618958910}},
                      {'key': None, 'value': {'customer_id': 24, 'view_time': 76, 'ts': 1786618968910}},
                      {'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

output_m_list = tn.process({click_source_str: click_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


<a id="min-operator"></a>
### min()

`min()` is syntactic sugar for `agg()` where the aggregation function calculates the minimum of the selected values:
```python
def min(self, value_fun, project_fun=lambda agg_any: agg_any, min_initial_any=sys.maxsize, **kwargs):
    """Minimum of values.
    
    Args:
        value_fun: r -> value_any - function to get the value to aggregate
        project_fun: (key_any, agg_any) -> r - projection function
        min_initial_any: initial minimum (default: sys.maxsize)
        **kwargs: passed through to the underlying node(s)
    Returns:
        tn: the newly created topology node of the operator"""
```

Here is an example:


In [ ]:
click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"]})
)

tn = Tn.build(
    click_tn
    ###
    # min() operator: 
    #   * select the value view_time
    #   * aggregation:
    #     * the minimum of the view times
    #   * projection: default = identity function of the aggregation
    ###
    .min(value_fun=lambda r: r["view_time"])
)

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618958910}},
                      {'key': None, 'value': {'customer_id': 24, 'view_time': 76, 'ts': 1786618968910}},
                      {'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

output_m_list = tn.process({click_source_str: click_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


<a id="avg-operator"></a>
### avg()

`avg()` is syntactic sugar for `agg()` where the aggregation function calculates the average of the selected values:
```python
def avg(self, value_fun, project_fun=lambda agg_any: agg_any, **kwargs):
    """Average of values.
    
    Args:
        value_fun: r -> value_any - function to get the value to aggregate
        project_fun: (key_any, agg_any) -> r - projection function
        **kwargs: passed through to the underlying node(s)
    Returns:
        tn: the newly created topology node of the operator"""
```

Here is an example:


In [ ]:
click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"]})
)

tn = Tn.build(
    click_tn
    ###
    # avg() operator: 
    #   * select the value view_time
    #   * aggregation:
    #     * the average of the view times
    #   * projection: default = identity function of the aggregation
    ###
    .avg(value_fun=lambda r: r["view_time"])
)

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618958910}},
                      {'key': None, 'value': {'customer_id': 24, 'view_time': 76, 'ts': 1786618968910}},
                      {'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

output_m_list = tn.process({click_source_str: click_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


<a id="count-operator"></a>
### count()

`count()` is syntactic sugar for `agg()` where the aggregation function counts the the selected values:
```python
def count(self, project_fun=lambda agg_any: agg_any, **kwargs):
    """Count records.
    
    Args:
        project_fun: (key_any, agg_any) -> r - projection function
        **kwargs: passed through to the underlying node(s)
    Returns:
        tn: the newly created topology node of the operator"""
```

Here is an example:


In [ ]:
click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"]})
)

tn = Tn.build(
    click_tn
    ###
    # count() operator: 
    #   * aggregation:
    #     * the count of the click records
    #   * projection: default = identity function of the aggregation
    ###
    .count()
)

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618958910}},
                      {'key': None, 'value': {'customer_id': 24, 'view_time': 76, 'ts': 1786618968910}},
                      {'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

output_m_list = tn.process({click_source_str: click_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


---
<a id="distinct-operator"></a>
## distinct()

The `distinct()` operator removes duplicate records, ensuring each unique record is represented only once. It has no counterpart in Kafka Streams.

Now you might ask why we would need such an operator in a relational stream processing engine working on sets anyway. Or why this operator is categorized as stateful and not stateless.

In DBSP/pydbsp, we actually work on streams of *deltas* of ZSets. A delta is a batch of records of any size.

If we just consider one delta, i.e., one batch of records, the relational, set-based nature of DBSP of course avoids duplicates natively. But if we receive another batch of records, DBSP can only detect whether it has already seen the individual records if it stores the previous delta/batch of records in its state.

To the syntax:
```python
def distinct(self, **kwargs):
    """Deduplicate.
    
    Args:
        **kwargs: passed through to the underlying node(s)
    Returns:
        tn: the newly created topology node of the operator"""
```

And to an example.

In [ ]:
click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"customer_id": r["value"]["customer_id"]})
)


non_distinct_sink_str = "non_distinct"
non_distinct_tn = click_tn.sink(non_distinct_sink_str)

distinct_sink_str = "distinct"
distinct_tn = (
    click_tn
    ###
    # distinct() operator
    ###
    .distinct()
    .sink(distinct_sink_str)
)

tn = Tn.build(non_distinct_tn, distinct_tn)

print("Step 1")

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618958910}},
                      {'key': None, 'value': {'customer_id': 4711, 'view_time': 76, 'ts': 1786618968910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

sink_str_output_m_list_dict = tn.process({click_source_str: click_input_m_list})
print("\nOutput:")
for sink_str, output_m_list in sink_str_output_m_list_dict.items():
    print(sink_str, output_m_list)

#

print("\nStep 2")

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

sink_str_output_m_list_dict = tn.process({click_source_str: click_input_m_list})
print("\nOutput:")
for sink_str, output_m_list in sink_str_output_m_list_dict.items():
    print(sink_str, output_m_list)


In the example, we first select only the `customer_id` of the inputs.

Then create two sinks:
* `non_distinct`: Here we do not use the `distinct()` operator.
* `distinct`: Here we do use it.

Here is the Mermaid version of the example topology:
```mermaid
graph TD
06451bcb-c1c4-4666-8d32-d3fb22da87b4[map_op] --> fd1e44b1-4dba-4cfa-87cb-b77fb15fb82d[sink_non_distinct]
fd1e44b1-4dba-4cfa-87cb-b77fb15fb82d[sink_non_distinct] --> 91075817-e169-48b5-af4c-68315e857610[sink_distinct]
7850acc4-8d90-4d5c-8050-56ffd99e9c99[map_op] --> 91075817-e169-48b5-af4c-68315e857610[sink_distinct]
3cab1cd5-4fbb-4006-88c9-d3c54aa74a99[source_clicks] --> 06451bcb-c1c4-4666-8d32-d3fb22da87b4[map_op]
abca870a-85ca-436e-a3a0-b65d528fbc14[distinct_op] --> 7850acc4-8d90-4d5c-8050-56ffd99e9c99[map_op]
06451bcb-c1c4-4666-8d32-d3fb22da87b4[map_op] --> abca870a-85ca-436e-a3a0-b65d528fbc14[distinct_op]
```

Now we feed in three messages with the same customer ID (=a duplicate output) in two steps:
1. We feed the first two messages. In the output you can see that both the `non_distinct` and the `distinct` sink are get the same result.
2. We feed the third message. Now the difference becomes visible:
  * `non_distinct`: Since this sink does *not* use the `distinct()` operator, it has forgotten that the third message is actually a duplicate and returns it.
  * `distinct`: For this sink, we *do* use the `distinct()` operator. And you can see that it has correctly detected the duplicate and hence does not return any output.


---
<a id="union-operator"></a>
## union()

The `union` operator takes two input streams and returns their union as in set theory:
```python
def union(self, other_tn, **kwargs):
    """Set union of two input streams.
        
    Args:
        other_tn: the other topology node to combine with
        **kwargs: passed through to the underlying node(s)
    Returns:
        tn: the newly created topology node of the operator"""
```

Once again, I think an example is the best explanation.

In [ ]:
click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"id": r["value"]["customer_id"]})
)

customer_tn = (
    Tn.source(customer_source_str)
    #
    .map(lambda r: {"id": r["value"]["id"]})
)

tn = Tn.build(
    click_tn
    ###
    # union() operator - get the set union of the outputs of click_tn and customer_tn
    ###
    .union(customer_tn)
)

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618958910}},
                      {'key': None, 'value': {'customer_id': 24, 'view_time': 76, 'ts': 1786618968910}},
                      {'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

customer_input_m_list = [{'key': '42', 'value': {'id': 42, 'name': 'Betty Graham MD'}},
                         {'key': '4711', 'value': {'id': 4711, 'name': 'Frank Frank'}}]
print("\nInput (customers):")
for m in customer_input_m_list:
    print(m)

output_m_list = tn.process({click_source_str: click_input_m_list, customer_source_str: customer_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


In the topology, we first select only the `customer_id` of the inputs and return it with the same field name `id`. Then we use the `union()` operator.

Here is the Mermaid version:
```mermaid
graph TD
32c98582-335c-480c-880c-f57a6efefb8a[map_op] --> 128251e1-9a05-4296-9e05-1ca008944fe0[union_op]
d03afce4-9980-48c1-b939-725b5ae70059[map_op] --> 128251e1-9a05-4296-9e05-1ca008944fe0[union_op]
0094590c-7a24-425a-9b26-3bb597d70197[source_clicks] --> d03afce4-9980-48c1-b939-725b5ae70059[map_op]
4b7964c2-8f31-4cdc-8fe9-3ef8d2936e95[source_customers] --> 32c98582-335c-480c-880c-f57a6efefb8a[map_op]
```

The two input sets are:
* from clicks: `{{'id': 42'}, {'id': 24}}`
* from customers: `{{'id': 42'}, {'id': 4711}}`

The union of these two sets is, as in the output from Kafi Streams:
```
{
    {'id': 42'}, 
    {'id': 24},
    {'id': 4711}
}
```


---
<a id="intersect-operator"></a>
## intersect()

The `intersect` operator takes two input streams and returns their intersection as in set theory:
```python
def intersect(self, other_tn, **kwargs):
    """Set intersection of two input streams.
    
    Args:
        other_tn: the other topology node to combine with
        **kwargs: passed through to the underlying node(s)
    Returns:
        tn: the newly created topology node of the operator"""
```

An example is coming up...

In [ ]:
click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"id": r["value"]["customer_id"]})
)

customer_tn = (
    Tn.source(customer_source_str)
    #
    .map(lambda r: {"id": r["value"]["id"]})
)

tn = Tn.build(
    click_tn
    ###
    # intersect() operator - get the set intersection of the outputs of click_tn and customer_tn
    ###
    .intersect(customer_tn)
)

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618958910}},
                      {'key': None, 'value': {'customer_id': 24, 'view_time': 76, 'ts': 1786618968910}},
                      {'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

customer_input_m_list = [{'key': '42', 'value': {'id': 42, 'name': 'Betty Graham MD'}},
                         {'key': '4711', 'value': {'id': 4711, 'name': 'Frank Frank'}}]
print("\nInput (customers):")
for m in customer_input_m_list:
    print(m)

output_m_list = tn.process({click_source_str: click_input_m_list, customer_source_str: customer_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


Here, we utilize the `intersect()` operator.

The graphical visualization is:
```mermaid
graph TD
06edc86a-2fc9-4fb1-b4fd-4d5158b55576[source_customers] --> 692dbf3b-464f-43ed-87d3-d0ab977e8225[map_op]
a0d7c437-df3a-45ea-a659-20f42d2ade43[source_clicks] --> f71a4a52-899a-464b-918f-cef5841e4180[map_op]
f71a4a52-899a-464b-918f-cef5841e4180[map_op] --> 8bf026bb-3bfb-4abf-9f9f-f158e0e3c896[intersect_op]
692dbf3b-464f-43ed-87d3-d0ab977e8225[map_op] --> 8bf026bb-3bfb-4abf-9f9f-f158e0e3c896[intersect_op]
```

The two input sets are (again):
* from clicks: `{{'id': 42'}, {'id': 24}}`
* from customers: `{{'id': 42'}, {'id': 4711}}`

The intersection of these two sets is, as in the output from Kafi Streams:
```
{
    {'id': 42'}, 
}
```


---
<a id="minus-operator"></a>
## minus()

The `minus` operator takes two input streams and returns their difference as in set theory:
```python
def minus(self, right_tn, **kwargs):
    """Set difference: self minus right_tn.
    
    Args:
        right_tn: the other topology node to subtract
        **kwargs: passed through to the underlying node(s)
    Returns:
        tn: the newly created topology node of the operator"""
```

To an example.


In [ ]:
click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"id": r["value"]["customer_id"]})
)

customer_tn = (
    Tn.source(customer_source_str)
    #
    .map(lambda r: {"id": r["value"]["id"]})
)

tn = Tn.build(
    click_tn
    ###
    # minus() operator - subtract the output from customer_tn from the output of click_tn
    ###
    .minus(customer_tn)
)

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618958910}},
                      {'key': None, 'value': {'customer_id': 24, 'view_time': 76, 'ts': 1786618968910}},
                      {'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

customer_input_m_list = [{'key': '42', 'value': {'id': 42, 'name': 'Betty Graham MD'}},
                         {'key': '4711', 'value': {'id': 4711, 'name': 'Frank Frank'}}]
print("\nInput (customers):")
for m in customer_input_m_list:
    print(m)

output_m_list = tn.process({click_source_str: click_input_m_list, customer_source_str: customer_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


Here we use the `minus()` operator:

```mermaid
graph TD
233af525-739c-466a-b2c2-1c3a2288001a[source_clicks] --> 9d0a249f-781c-40a2-9110-65ff58133cfd[map_op]
d567170b-0615-4f65-acf1-aaa6ab5dc8f0[source_customers] --> c80c0463-a693-4665-8a82-de8fab2073cc[map_op]
c80c0463-a693-4665-8a82-de8fab2073cc[map_op] --> 1ba2f222-e6a9-49a3-add9-78e4123e7c0b[minus_op]
9d0a249f-781c-40a2-9110-65ff58133cfd[map_op] --> 1ba2f222-e6a9-49a3-add9-78e4123e7c0b[minus_op]
```

The two input sets are (again):
* from clicks: `{{'id': 42'}, {'id': 24}}`
* from customers: `{{'id': 42'}, {'id': 4711}}`

The set difference of these sets from clicks and customeers is, as in the output from Kafi Streams:
```
{
    {'id': 24'}
}
```


---
<a id="collapse-operator"></a>
## collapse()

The `collapse` operator collapses changelog/upsert streams by key and handles tombstone messages. This operator is key for handling the [stream/table duality](../duality.ipynb) for changelog topics:
```python
def collapse(self, key_fun=lambda r: r["key"], is_deletion_fun=lambda r: r["value"] is None, **kwargs):
    """Collapse/resolve a changelog stream (inserts/updates/deletes per key) down to the latest value per key, retracting the previous value whenever a new value (or a deletion) arrives for that key.

    Args:
        key_fun: r -> key - the key to resolve on (default: lambda r: r["key"])
        is_deletion_fun: r -> bool - True if r represents a delete (default: r["value"] is None, aka Kafka tombstone messages)
        **kwargs: passed through to the underlying node(s)
    Returns:
        tn: the newly created topology node of the operator"""
```

Here is an example.


In [9]:
sink_tn = (
    Tn.source(customer_source_str)
    ###
    # collapse() operator - collapse changelog streams by key
    ###
    .collapse()
    #
    .group_by_count(
        key_fun=lambda r: r["value"]["id"],
        project_fun=lambda key_any, agg_any: {"id": key_any, "count": agg_any})
)

tn = Tn.build(sink_tn)
tn.from_zSet(Tn._to_records)

print("Step 1")

customer_input_m_list = [{'key': '42', 'value': {'id': 42, 'name': 'Betty Graham MD'}},
                         {'key': '4711', 'value': {'id': 4711, 'name': 'Frank Frank'}}]
print("\nInput (customers):")
for m in customer_input_m_list:
    print(m)

output_m_list = tn.process({customer_source_str: customer_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)

print("\n---")
print("\nStep 2")

customer_input_m_list = [{'key': '42', 'value': {'id': 42, 'name': 'Betty G. Miller'}}]
print("\nInput (customers):")
for m in customer_input_m_list:
    print(m)

output_m_w_tuple_list = tn.process({customer_source_str: customer_input_m_list})
print("\nOutput:")
for m_w_tuple in output_m_w_tuple_list:
    print(m_w_tuple)

print("\n---")
print("\nStep 3")

customer_input_m_list = [{'key': '42', 'value': None}]
print("\nInput (customers):")
for m_w_tuple in customer_input_m_list:
    print(m_w_tuple)

output_m_w_tuple_list = tn.process({customer_source_str: customer_input_m_list})
print("\nOutput:")
for m_w_tuple in output_m_w_tuple_list:
    print(m_w_tuple)

print("\n---")
print("\nStep 4")

customer_input_m_list = [{'key': '42', 'value': {'id': 42, 'name': 'Betty G. Miller'}}]
print("\nInput (customers):")
for m_w_tuple in customer_input_m_list:
    print(m_w_tuple)

output_m_w_tuple_list = tn.process({customer_source_str: customer_input_m_list})
print("\nOutput:")
for m_w_tuple in output_m_w_tuple_list:
    print(m_w_tuple)


Step 1

Input (customers):
{'key': '42', 'value': {'id': 42, 'name': 'Betty Graham MD'}}
{'key': '4711', 'value': {'id': 4711, 'name': 'Frank Frank'}}

Output:
({'id': 42, 'count': 1}, 1)
({'id': 4711, 'count': 1}, 1)

---

Step 2

Input (customers):
{'key': '42', 'value': {'id': 42, 'name': 'Betty G. Miller'}}

Output:

---

Step 3

Input (customers):
{'key': '42', 'value': None}

Output:
({'id': 42, 'count': 1}, -1)

---

Step 4

Input (customers):
{'key': '42', 'value': {'id': 42, 'name': 'Betty G. Miller'}}

Output:
({'id': 42, 'count': 1}, 1)


Here, we use the `collapse()` operator right after the source specification to handle the incoming changelog stream. Under the covers, it keeps a mapping of keys to values and retracts records once a new record with the same key comes in. With `is_deletion_fun` deletions can also be handled - if a deletion is detected (typically, a Kafka tombstone record where the `value` is `None`), the record to be deleted is looked up and it gets weight `-1` to mark it as deleted for Kafi Streams/pydbsp.

Then, the `group_by_count()` operator is used to count the customers by their ID:
```mermaid
graph TD
12c6b2d7-141d-4165-9a5f-157fecd705de[source_customers] --> 80d0a3a0-7c91-4e02-9b90-d04cb43709c3[collapse_op]
80d0a3a0-7c91-4e02-9b90-d04cb43709c3[collapse_op] --> abc08307-1a95-47ff-8836-df8d80ae413b[group_by_count_op]
```

In the example, we go through four steps:
1. We insert two customers. Both then have `count = 1`.
2. We update customer `42`. The counts stay unchanged, hence the output is empty.
3. We delete customer `42`. The count for this customer is retracted (=weight `-1`).
4. We insert customer `42` once again. Its count goes up to `1` again.
